# Zyro Dynamics HR Help Desk — RAG Challenge
### NxtWave Masterclass | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Deploy a Streamlit chatbot
- Generate your `submission.csv`

## Submission Requirements

1. `submission.csv` — upload on Kaggle
2. Streamlit App URL
3. LangSmith Trace URL

---

> Follow the notebook cells sequentially and complete the sections marked for implementation.

## Cell 1 — Install Dependencies

> ⚠️ Run this cell first before anything else.

This cell installs all required libraries for:
- document loading
- embeddings
- vector database
- RAG pipeline
- Streamlit deployment
- LangSmith tracing

> After installation completes, restart the kernel/runtime and run all cells from the top.

In [1]:
print("Installing required packages...\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    streamlit \
    langsmith \
    python-dotenv \
    tiktoken

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

Installing required packages...


Installation complete.
Please restart the kernel/runtime before running the next cell.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2 — Configuration

This is the main configuration cell for the notebook.

Here you can:
- choose your LLM provider
- select the model you want to use
- update related settings if needed

All remaining cells will automatically use this configuration.

In [2]:
LLM_PROVIDER = "groq" # "groq" | "gemini" | "openai"
LLM_MODEL = "llama-3.1-8b-instant" # change model here if needed

CORPUS_PATH = "/kaggle/input/zyro-dynamics-hr-corpus/"

print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")

Provider: groq
Model: llama-3.1-8b-instant


## Cell 3 — Imports

This cell imports all required libraries for:
- document loading
- text chunking
- embeddings
- vector search
- prompt handling
- LangSmith tracing

> Run this cell without modifying anything.

In [3]:
import os, json, time, csv
from cryptography.fernet import Fernet
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langsmith import traceable

print("Imports loaded successfully.")

C:\Users\Guru Mahesh\AppData\Local\Temp\ipykernel_24372\3671248947.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\Guru Mahesh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\Guru Mahesh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded successfully.


## Cell 4 — API Keys + LangSmith Setup

This cell loads:
- your LLM API key
- LangSmith API key
- environment configuration

LangSmith tracing is enabled automatically for monitoring and debugging your RAG pipeline.

> Add the required API keys before running this cell.
> This section is pre-filled — no modifications needed.

In [6]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    if LLM_PROVIDER == "groq":
        os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
    elif LLM_PROVIDER == "gemini":
        os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    elif LLM_PROVIDER == "openai":
        os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")

    os.environ["LANGCHAIN_API_KEY"]    = secrets.get_secret("LANGCHAIN_API_KEY")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"
    print("Running on Kaggle — secrets loaded!")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

print("Environment configured successfully.")

Environment configured successfully.


## Cell 5 — Load Documents

### Your Task

Load all policy documents from the provided corpus directory using a PDF loader.

Store the loaded documents and print the total number loaded.

In [7]:
# Try downloading from Kaggle helper first (if available)
corpus_path = None
try:
    import kagglehub
    comp_path = kagglehub.competition_download('niat-masterclass-rag-challenge')
    if comp_path and os.path.exists(comp_path):
        comp_dir = comp_path if os.path.isdir(comp_path) else os.path.dirname(comp_path)
        candidate = os.path.join(comp_dir, 'zyro-dynamics-hr-corpus')
        if os.path.exists(candidate) and any(f.lower().endswith('.pdf') for f in os.listdir(candidate)):
            corpus_path = candidate
        elif any(f.lower().endswith('.pdf') for f in os.listdir(comp_dir)):
            corpus_path = comp_dir
except Exception:
    pass

# If kagglehub didn't provide the corpus, try common local locations
if corpus_path is None:
    candidates = [
        CORPUS_PATH,
        os.path.join(os.getcwd(), "zyro-dynamics-hr-corpus"),
        os.getcwd(),
        os.path.join(os.path.dirname(os.getcwd()), "zyro-dynamics-hr-corpus"),
    ]

    for p in candidates:
        try:
            if p and os.path.exists(p) and any(f.lower().endswith('.pdf') for f in os.listdir(p)):
                corpus_path = p
                break
        except Exception:
            continue

# Fallback: scan workspace for the first directory that contains PDFs
if corpus_path is None:
    for root, dirs, files in os.walk(os.getcwd()):
        if any(f.lower().endswith('.pdf') for f in files):
            corpus_path = root
            break

if corpus_path is None:
    raise FileNotFoundError(
        "No PDF files found in the workspace or candidate paths.\n"
        f"Tried: kagglehub, {candidates if 'candidates' in locals() else 'none'}\n"
        "Please place the PDF corpus in the workspace or update `CORPUS_PATH`."
    )

# Initialize document loader
loader = PyPDFDirectoryLoader(corpus_path)

# Load documents
documents = loader.load()

# Display document breakdown
print(f"Loaded {len(documents)} document pages from {corpus_path}\n")
print("Breakdown by PDF file:")
pdf_pages = {}
for doc in documents:
    source = doc.metadata.get('source', 'Unknown')
    pdf_name = os.path.basename(source)
    pdf_pages[pdf_name] = pdf_pages.get(pdf_name, 0) + 1

for pdf_file in sorted(pdf_pages.keys()):
    print(f"  - {pdf_file}: {pdf_pages[pdf_file]} page(s)")

print(f"\nTotal PDFs: {len(pdf_pages)}")
print(f"Total pages (documents): {len(documents)}")


Loaded 39 document pages from c:\Users\Guru Mahesh\Downloads\niat-masterclass-rag-challenge\zyro-dynamics-hr-corpus

Breakdown by PDF file:
  - 00_Company_Profile.pdf: 4 page(s)
  - 01_Employee_Handbook.pdf: 4 page(s)
  - 02_Leave_Policy.pdf: 4 page(s)
  - 03_Work_From_Home_Policy.pdf: 3 page(s)
  - 04_Code_of_Conduct.pdf: 3 page(s)
  - 05_Performance_Review_Policy.pdf: 4 page(s)
  - 06_Compensation_and_Benefits_Policy.pdf: 4 page(s)
  - 07_IT_and_Data_Security_Policy.pdf: 3 page(s)
  - 08_Prevention_of_Sexual_Harassment_Policy.pdf: 3 page(s)
  - 09_Onboarding_and_Separation_Policy.pdf: 4 page(s)
  - 10_Travel_and_Expense_Policy.pdf: 3 page(s)

Total PDFs: 11
Total pages (documents): 39


## Cell 6 — Chunk Documents

### Your Task

Split the loaded documents into smaller chunks using `RecursiveCharacterTextSplitter`.

Store the generated chunks and print the total number created.

In [8]:
# Initialize text splitter with optimal chunk size and overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=768,
    chunk_overlap=128,
    separators=[
        "\n\n\n",
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ]
)

# Create document chunks
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunk size: {sum(len(chunk.page_content) for chunk in chunks) // len(chunks) if chunks else 0} characters")


Created 115 chunks from 39 documents
Average chunk size: 637 characters


## Cell 7 — Embeddings

### Your Task

Initialize an embedding model to convert document chunks into vector representations.

Use `HuggingFaceEmbeddings` for the implementation below.

In [9]:
import torch

# Choose GPU when available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model initialized.")


Using device: cpu


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3337.36it/s]


Embedding model initialized.


## Cell 8 — Vector Store + Retriever

### Your Task

Build a FAISS vector store using the generated chunks and embeddings.

Then create a retriever for retrieving relevant document chunks during question answering.

In [10]:
# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"FAISS index created successfully!")
print(f"Total vectors indexed: {vectorstore.index.ntotal}")

# Create an optimized retriever with MMR
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 15,
        "lambda_mult": 0.7
    }
)

print("Retriever initialized.")


FAISS index created successfully!
Total vectors indexed: 115
Retriever initialized.


## Cell 9 — LLM Initialization

The language model is initialized automatically using the configuration from Cell 2.

You may optionally adjust parameters such as:
- `temperature`
- `max_tokens`

based on your preferred response style and performance.

> This cell is pre-filled — modify only if needed.

In [11]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_output_tokens=512
    )

elif LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=512
    )

else:
    raise ValueError("Unsupported LLM provider.")

print("LLM initialized.")

LLM initialized.


## Cell 10 — Build the RAG Chain

### Your Task

Implement the RAG pipeline using LCEL and enable LangSmith tracing.

In [12]:

# Prompt Template
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an HR Policy Assistant.

Use ONLY the provided context to answer the question.

Rules:
1. Answer only from the retrieved context.
2. If the answer is not present in the context, respond:
   "I could not find that information in the HR policies."
3. Do not make assumptions or hallucinate information.
4. Be concise and professional.
5. When possible, mention the policy source.

Context:
{context}

Question:
{question}

Answer:
""")


# Format retrieved documents
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'Unknown')}, "
        f"Page: {doc.metadata.get('page', 'Unknown')}]\n"
        f"{doc.page_content}"
        for doc in docs
    )


# LCEL RAG Chain
rag_pipeline = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)


# Wrapper Function
def rag_chain(question: str):
    return rag_pipeline.invoke(question)


print("RAG pipeline initialized.")

RAG pipeline initialized.


## Cell 11 — Guardrails

### Your Task

Implement handling for out-of-scope questions before routing queries through the RAG pipeline.

In [13]:

OOS_PROMPT = ChatPromptTemplate.from_template("""
You are a classifier.

Determine whether the question is related to employee HR policies.

HR policy topics include:
- Employee handbook
- Leave policy
- Work from home
- Compensation and benefits
- Performance reviews
- Code of conduct
- IT and data security
- Travel and expenses
- Onboarding
- Separation
- Sexual harassment policy
- Company policies

Respond with ONLY:

YES
or

NO

Question:
{question}
""")

# Build classifier chain
oos_classifier = (
    OOS_PROMPT
    | llm
    | StrOutputParser()
)

REFUSAL_MESSAGE = (
    "I'm sorry, but I can only answer questions related to the company's HR policies and documents."
)

def ask_bot(question: str):

    classification = oos_classifier.invoke(
        {"question": question}
    ).strip().upper()

    if classification != "YES":
        return REFUSAL_MESSAGE

    return rag_chain(question)

print("Guardrails initialized.")

Guardrails initialized.


## Cell 12 — Test the Bot

### Your Task

Test your RAG pipeline using a few sample questions of your choice.

In [14]:
test_questions = [
    "How many annual leaves are employees entitled to each year?",
    "What is the company's work from home policy?",
    "What should an employee do if they experience workplace harassment?",
    "What expenses can employees claim during business travel?",
    "What is the performance review process?",
    "Who won the FIFA World Cup in 2022?"  # Out-of-scope test
]

for i, q in enumerate(test_questions, 1):
    print(f"Q{i}: {q}")

    # Run chatbot
    response = ask_bot(q)

    print("\nAnswer:")
    print(response)

    print("-" * 60)

Q1: How many annual leaves are employees entitled to each year?

Answer:
I could not find that information in the HR policies.
------------------------------------------------------------
Q2: What is the company's work from home policy?

Answer:
According to the Work From Home Policy (Doc Code: ZDL-HR-003), Zyro Dynamics Pvt. Ltd. recognises the importance of flexible work arrangements for employee wellbeing, productivity, and retention. The policy establishes a structured framework for employees to work from a location other than their designated Zyro Dynamics office, with the expectation that all WFH arrangements will be exercised responsibly and that the quality and timeliness of work will not be compromised.

Source: c:\Users\Guru Mahesh\Downloads\niat-masterclass-rag-challenge\zyro-dynamics-hr-corpus\03_Work_From_Home_Policy.pdf, Page 1.
------------------------------------------------------------
Q3: What should an employee do if they experience workplace harassment?

Answer:
Acc

## Cell 13 — LangSmith Trace URL

Generate and copy your shareable LangSmith trace URL for submission.

> This cell is pre-filled — just run it and follow the instructions.

In [15]:
print("""
HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!
""")


HOW TO GET YOUR LANGSMITH TRACE URL
════════════════════════════════════
1. Go to: https://smith.langchain.com
2. Sign in with your account
3. Click on project: 'zyro-rag-challenge'
4. You will see all your RAG traces here
5. Top right corner → Share → Enable Public Link
6. Copy the URL
7. Paste this URL when running Cell 16!



## Cell 14 — Streamlit App

### Your Task

Build and deploy a Streamlit chatbot application for your RAG pipeline.

Save your implementation as `app.py`.

In [ ]:
app_code = """
# TODO: Build your Streamlit chatbot application

import streamlit as st

# your code here
"""

with open("app.py", "w") as f:
    f.write(app_code.strip())

print("app.py created.")

app.py created.


app.py created.


## Cell 15 — Evaluation

Evaluation inputs are loaded automatically at runtime.

> Do not modify this cell.

In [ ]:
_Q = [
    ("Q01", "gAAAAABqE-m-EnBhR94RLAsyCD5YUOimCgpyxnGmrg3N29dvcCChh_LbQzGhacqtB6Rg9ySTN-aO4eS5nnSSqgvslxWg3T2XNxvKRw9BoZOGB8sSrPpeXOqPKhdprAkvepa0Ef13rK84Lx_QKNPq5AMeO2zweDFo-UGpOZ1yFV_k0NbpkP0MshR9BpjCI4QpkDSx9QH95aeCK8sqSIkcM8wOFRs1hRD_tV-Jg4XmeHLm4jW6wpCWQRBF-XWIHTwCE3Tod-Zfj-nIFpPe3sNmXFDNY_L5g8aAiw=="),
    ("Q02", "gAAAAABqE-m-iGIUkxaPu-TWqkoQqfrY1QvCn-VC445z8EzeRjBVVSjcBgTYC-OS2QVoM37Oh8tFkJdLJcdivCIg9-jTJ72Vy24BQwagKYrIJlkNBr9yectRVtDZ_X24PWpsbIdMcelH1a6VBz9XXmJ19-0HvqFT0kTeEQEyjzKL2BmtoSHOquqe74xGFhpWD-fI1Cshfxk9EXwgA4poqi7JJ3ovja5pVM18uwfNAmcNacnQRtFTAm6x1JmXKSYVeBSbgpOv1zjEEC-0XfVhF0Wtwli0hRZHhA=="),
    ("Q03", "gAAAAABqE-m-qhjI3OCH68smnD4afuA_GmeOO8rI6R79iaPeodfwbt4NTlWhlbSfgr8BP9ZNAi5yczk65fgsIgbRXQ9AkAVDE2NOD11Aqt6U_NqURkjBQpzn3gzTQNj2qNwtkhx71-l8uYIfZLu8Z-Nv4aAkEaFTKCDp4DWgCaFJbe90TCA2fGUVnDiaI1_0ID87AHR-yYRwTaKYiWI7PiCQWFVm22NGx3cwX_uvMouAEXLX2sw_o3s="),
    ("Q04", "gAAAAABqE-m-qVKLekYizIYVBejJAmZYhT0zftdQzC0nbFt6BAJM52tiRsM0y5pcEfTl7y2bKwjFBSBwj3ik1P1yPTz6mP2h1xHEWoeJxPHdvujlZXJv8ObZO0PbHSPMk6xtnEmEqPAfPLzxjOzu63P3K_0eFdpgR48fUbcQwZt7yZkGzOPqYuUDAE7CBmvgvwRfwymkEzTD8ESt0vYvZdmoYjV7sbScmhoxYbWmjMatFvOzha6D1YA="),
    ("Q05", "gAAAAABqE-m-KRbrY2MpEseeszU46iQWHzbzwOO5-t10vHJrdQOKeaVwPxyp9kiBDCS1Fa5MJyQoTOp2pdEtw9LtUbCEJ_56caOBjtBgngLz4kvcodhVECBLBuD6vsCaQlopu0SardsvA3slA379M8nrcyuuea3dJ97FPlOdQs2b70BRPyOkyNH0nKGqBwQzBlAW7B-ucZwf9dDPPAw-xUTfR3ekIqXReQ=="),
    ("Q06", "gAAAAABqE-m-EYfgWBpxkb_5hGOvvBsAdBu5367Nd5d4uT_6EEAaTeCidG99u5XJ5vcZatZpoj5RjmfrY5O1XNObuApuq_ZFah_StEcLHB31Ow6WRrZpikDGUFJkC-ZfY0TggJzDFvdtwQsIttqNW5js0LMS-74V-AUx0UCi4bABm1vOMGBKP2qGyGTfyh2wfETTw4nNhbac"),
    ("Q07", "gAAAAABqE-m-cZLyG6To-HyWWdEYu42VgbV9c_SCWXt4qJE02YrOFvfMntuBTf-CVXt3MhJWFzrukGMR0-Brla1QMVbefRelzpJqkY2TsIQ3Tcc5MZ0BH6ornHjZAnOd9Iozf1f755EC8hBase1XtbhThrKgYJRKWPxaxKd-nkLK3XuabtmEF8r0bZtTyKVjYNBUWPT--lKJb-pXvw3p3zJ0z6utBLWicmBhgdJvGMoOQCsCLrxi6jrtHZzka7Me7Vm6UUhwSkdz"),
    ("Q08", "gAAAAABqE-m-sxXijCcjguEWTh7qgKt7BX4cbUfFdUwAz6VqSoU4fTnYXUhf-dVQdCKa1lhgc7ZZatU5Pu9iuQHG-ApZCOw2yR-PkZnuY9L7uR02CCJoWYhFQelqYEWYA5uONridoCzD8kh2yqwUSVInEFfBuB2cYgyPobRnP_yRvtaFtLakrMy0fsCZH_zfyrOMVkdF5GoHdPu67XzoEj806x4aS8DJ4ysYFuwNb9zkhhceq_CsU08="),
    ("Q09", "gAAAAABqE-m-nDGYgCF3fSWs2tM39pdnsBua61Ht1ruTZ_NOUmju6AxbGU6WB8HzLEHKQkkCnxc4ka2DohiUSLwVDrWG2ZnGggyt7OnI6D43ovjDBsMhW2jQPaz9zaHua25abfEqF4V1ZioQrdL7lz3D0qzDsjXl4Kw5RY2g3kaDakb62Cb6Dt8badoS-t4Bd_fEAp49t09FH_qwLp_ZTotiFsKFy6QADA=="),
    ("Q10", "gAAAAABqE-m-PwoVsLjWO4nbO8W_65P-UNNF7SjdNZL4sRN-G72eHygPuGyggXwVG8G7HJ2ZmrtCYuNg-rtWH_iuyexPQLVG0EqKr0ZQswJox4iauvFf014qlqr5vC_TtdwHGcMiZsyWZpJauDTffKDm_QJHrGElPUUunCFgX8356s1yMocleGXUBfcZ8B73A5LIALAXRIBpKyt707qYlLhwOG1vhsdR74q21NS0-n0skLZIy7z0pLM="),
    ("Q11", "gAAAAABqE-m-1BAGkhsZEDnkbSwAAwusmnMKdn2gvIM5tltaZ1W-eoKtvbPNu8rkAlOOiOW-9_NobJqDFKDO3J7zCPwWuEdGxwgYpX5sxh2Rg4ngR5R5WDnQsQTPIRHXJkkaN1ufNhvbQ-XOn2Z1QPci8118ByVpkAR5kZTUXOFIZ1IgHP2hbvO4E81GB9CTs9HiZvHAsAnS"),
    ("Q12", "gAAAAABqE-m-NrwI-KspXny9JlQqBEW_eB9jE6bGmnin6IX6SdcB9ol1gR7CmzczDKE6A7XHDOJW20tVHAlGFw-q-J6cWrTajK_mJTv00aHllSozrKiThojuxxnSjhgOhgtNKU5mh7zoz2d2uLo7p-Kl32m4IU6PRsm0kZceID-ZH5ZRw7w4h1qSZOufZO2HvKkR9LtfCQXk"),
    ("Q13", "gAAAAABqE-m-Xr56G8qaFfk3BIVQeDzP5mpahd7wZQ5vGR11AN_sxU1ZzjoPfbSdLmrrhFHEI8S8KhXfjOWZQoMJToWSsnhjZQdrRj0wujH38p2VOZLqqZYSmOflVEQm29z9pAXx_iltLWZLNGf8QsMtZWuo-3SsWt6R2mGvOMBTDj5hCzaq842_r1eupRQJJ1dnTSmNPskW"),
    ("Q14", "gAAAAABqE-m--oxJAL26EQ6bMS5vmgI0pWMWjgbG49qNZu8K_pIiDrp3ro1YFlVvBXOOJ6bSpI7lxz-OXmNrVFkSfJlVf4PchVKfWdddKVT85AMxUHo3PYD15IGV476RznHCiD59twp7x_E6HOF7AFUGiWcsO9Ph63Tfcvh3aJzF7Hk_NPEHcIaaEU9ki2eccYXehJJ3tkmr"),
    ("Q15", "gAAAAABqE-m-3JNAfb2dmCF-2XlNe-F1AaeXybgSJ4DwHtn9o52TEryPYgu-6m70Ivn7izeLy4h44AVbHL_3cv-MWfAwFYp7ct3lvF7dL1QbmhntyeY4c7l0CVPsc-mv8WuY04tpB2XPtHE_0ytl9tQlqAGonC2esnpMbSzgvZPdSw9eHnm5k2Jkh0FbgjLKNWxjdX3Uv2aYDiqOeLMQKZsMMteZzJcwHQ=="),
]

eval_questions = [
    {"question_id": qid, "question": fernet.decrypt(enc.encode()).decode()}
    for qid, enc in _Q
]

print(f"{len(eval_questions)} evaluation questions loaded.")

15 evaluation questions loaded.


## Cell 16 — Generate `submission.csv`

Generate your final `submission.csv` file for submission.

> Do not modify this cell.

In [ ]:
import re

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("=" * 50)
print("Submission Generator")
print("=" * 50)

streamlit_link = input("Streamlit App URL: https://zyro-dynamics-hr-app-desk-vhmn5jh5qnkggzxqnjxewq.streamlit.app").strip()
langsmith_link = input("LangSmith Trace URL: https://smith.langchain.com/public/9a5825ba-d203-4bc0-9db4-e12aa1960f6b/r").strip()

link_errors = []

if not STREAMLIT_PATTERN.match(streamlit_link):
    link_errors.append("Invalid Streamlit URL.")

if not LANGSMITH_PATTERN.match(langsmith_link):
    link_errors.append("Invalid LangSmith URL.")

if link_errors:
    print("\n".join(link_errors))
    raise ValueError("Please correct the links and re-run the cell.")

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator


## Cell 17 — Final Checklist

Verify your submission files and links before submitting on Kaggle.

> This cell is pre-filled — just run it.

In [ ]:
import re, csv, os

STREAMLIT_PATTERN = re.compile(
    r"^https://.+\.streamlit\.app(/.*)?$",
    re.IGNORECASE
)

LANGSMITH_PATTERN = re.compile(
    r"^https://smith\.langchain\.com/.+",
    re.IGNORECASE
)

print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):

    with open("submission.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    count = len(rows)

    has_fields = all(
        all(
            k in r
            for k in [
                "question_id",
                "question_enc",
                "answer_enc",
                "streamlit_link",
                "langsmith_link"
            ]
        )
        for r in rows
    )

    sl_valid = all(
        STREAMLIT_PATTERN.match(r["streamlit_link"].strip())
        for r in rows
    )

    ll_valid = all(
        LANGSMITH_PATTERN.match(r["langsmith_link"].strip())
        for r in rows
    )

    print(f"submission.csv found ({count} rows)")
    print(f"Required columns present: {has_fields}")
    print(f"Streamlit links valid: {sl_valid}")
    print(f"LangSmith links valid: {ll_valid}")

    if not sl_valid or not ll_valid:
        print("\nPlease regenerate submission.csv with valid links.")

else:
    print("submission.csv not found. Run the previous cell first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")